# Notebook 04 - Livability Scoring Engine (M5)

Computes livability scores for all places, runs sensitivity analysis, exports `scores_wide.csv`.

References: M5.1-M5.7

In [ ]:
import sys, os
ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))
if ROOT not in sys.path:
    sys.path.insert(0, ROOT)
import warnings; warnings.filterwarnings("ignore")
import pandas as pd, numpy as np, matplotlib.pyplot as plt, seaborn as sns
from src.utils.helpers import setup_logging
setup_logging()
sns.set_theme(style="whitegrid")
FIGURES = "../reports/figures"
os.makedirs(FIGURES, exist_ok=True)
print("Environment ready.")
from src.utils.constants import FACTORS_WIDE, PLACES_CSV, SCORES_WIDE, SCORING_CONFIG
from src.scoring.scoring import load_config, compute_scores, sensitivity_analysis, normalize_value
tidy = None
wide = pd.read_csv(FACTORS_WIDE)
places = pd.read_csv(PLACES_CSV)
config = load_config(SCORING_CONFIG)
print('Factors in config:', list(config['factors'].keys()))
print('Wide shape:', wide.shape)

## 1. Normalization demo (M5.2)

In [ ]:
print('lower_is_better, bounds=[10, 150]:')
for v in [10, 80, 150, 200]:
    print(f'  PM2.5={v} -> score={normalize_value(v, 10, 150, "lower_is_better"):.1f}')

## 2. Compute scores with balanced weights

In [ ]:
weights = {'air_quality': 0.30, 'safety': 0.25, 'health': 0.25, 'economy': 0.20}
scored = compute_scores(wide, config, weights, year=None)
print(f'Scored {len(scored)} places')
display(scored[['place_id','year','score','rank','partial_data','missing_factors']].head(20))

## 3. Top 20 ranking

In [ ]:
scored_named = scored.merge(places[['place_id','place_name','place_type','region']], on='place_id', how='left')
top20 = scored_named.dropna(subset=['score']).nsmallest(20, 'rank')
fig, ax = plt.subplots(figsize=(10,6))
import matplotlib.cm as cm
colors = cm.RdYlGn(top20['score'].values / 100)
ax.barh(top20['place_name'], top20['score'], color=colors)
ax.set_xlabel('Livability Score (0-100)')
ax.set_title('Top 20 Places — Balanced Weights')
ax.invert_yaxis()
ax.set_xlim(0, 100)
plt.tight_layout()
plt.savefig(f'{FIGURES}/04_top20_balanced.png', dpi=150, bbox_inches='tight')
plt.show()

## 4. Factor score breakdown

In [ ]:
factor_cols = [c for c in scored.columns if c.endswith('_score')]
if factor_cols:
    top10 = scored_named.dropna(subset=['score']).nsmallest(10, 'rank')
    breakdown = top10[['place_name'] + factor_cols].set_index('place_name')
    breakdown.columns = [c.replace('_score','').replace('_',' ').title() for c in breakdown.columns]
    breakdown.plot(kind='bar', figsize=(12,5), colormap='tab10', edgecolor='white')
    plt.title('Factor Score Breakdown - Top 10 Places')
    plt.ylabel('Factor Score (0-100)')
    plt.xticks(rotation=30, ha='right')
    plt.legend(bbox_to_anchor=(1.01,1), loc='upper left', fontsize=9)
    plt.tight_layout()
    plt.savefig(f'{FIGURES}/04_factor_breakdown.png', dpi=150, bbox_inches='tight')
    plt.show()

## 5. Compare preset profiles

In [ ]:
presets = {'Balanced': {'air_quality':0.30,'safety':0.25,'health':0.25,'economy':0.20}, 'Clean Air': {'air_quality':0.55,'safety':0.15,'health':0.20,'economy':0.10}, 'Family': {'air_quality':0.20,'safety':0.30,'health':0.30,'economy':0.20}, 'Career': {'air_quality':0.15,'safety':0.15,'health':0.10,'economy':0.60}, 'Retirement': {'air_quality':0.30,'safety':0.15,'health':0.45,'economy':0.10}}
rank_rows = []
for preset_name, w in presets.items():
    res = compute_scores(wide, config, w, year=None)
    res['preset'] = preset_name
    rank_rows.append(res[['place_id','rank','preset']])
rank_df = pd.concat(rank_rows, ignore_index=True)
rank_pivot = rank_df.merge(places[['place_id','place_name']], on='place_id').pivot(index='place_name', columns='preset', values='rank')
display(rank_pivot.dropna().nsmallest(15, 'Balanced'))

## 6. Sensitivity analysis (M5.6)

In [ ]:
print('Running 1000 Dirichlet draws...')
sens = sensitivity_analysis(wide, config, year=None, n_draws=1000)
sens = sens.merge(places[['place_id','place_name']], on='place_id', how='left')
print(f'Robust places (rank_range <= 3): {sens["robust"].sum()}')
display(sens[['place_name','median_rank','min_rank','max_rank','rank_range','robust']].head(20))

## 7. Export scores_wide.csv

In [ ]:
from src.utils.constants import DATA_PROCESSED
DATA_PROCESSED.mkdir(parents=True, exist_ok=True)
scored_named.to_csv(SCORES_WIDE, index=False)
print(f'Saved: {SCORES_WIDE}')